# CineFusion: BellKor Ensemble (CUR + CBF + SVD)

Combines three base recommenders into a learned linear blender, like the Netflix Prize's Bellkor

1. **CUR** - sketched matrix factorization (loaded from `CineFusion CUR.ipynb` checkpoints).
2. **SVD** - biased truncated SVD on the same centered training matrix (trained here).
3. **CBF** - user-profile content scorer built from BAAI/BGE TMDB embeddings.

A ridge regression learns blend weights on a held-out half of the test set; the
other half is used for a fair side-by-side evaluation against each base model.


## Step 0: Setup

In [ ]:
import os, gc, json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.sparse import load_npz, csr_matrix
from scipy.sparse.linalg import svds
from sklearn.linear_model import Ridge

# --- Paths (mirrors the CUR file I did) ---
BASE_DIR    = "."                                        # run from project root
CKPT_DIR    = f"{BASE_DIR}/cur-output/cur_checkpoints"   # CUR notebook outputs
SVD_DIR     = f"{BASE_DIR}/cur-output/svd_checkpoints"   # biased SVD factors
OUTPUT_DIR  = f"{BASE_DIR}/cur-output"                   # blender weights + eval json
EMB_PATH    = f"{BASE_DIR}/tmdb-5000-embeddings-BAAI/embeddings.npy"
LINKS_PATH  = f"{BASE_DIR}/ml-25m/links.csv"
TMDB_PATH   = f"{BASE_DIR}/tmdb-5000/tmdb_5000_movies.csv"
MOVIES_PATH = f"{BASE_DIR}/ml-25m/movies.csv"

os.makedirs(SVD_DIR,    exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Hyperparams ---
SEED        = 42                # match CUR notebook
SVD_RANK    = 50                # match CineFusion_SVD.ipynb ALS rank for fair comparison
SVD_BIAS_LAMBDA = 25.0          # regularizer for per-item bias (Koren 2009 default style)
RIDGE_ALPHA = 1.0               # blender L2
EVAL_USERS  = 5000              # subsample for ranking metrics (full eval is 80k+ users)
K           = 10                # top-K for ranking metrics
RELEVANCE_THRESHOLD = 4.0
MIN_RELEVANT        = 2
MIN_SUPPORT         = 20        # min train ratings per movie to be eligible (matches CUR)
NUM_NEGATIVES       = 99        # for HitRate/NDCG sampled eval

# --- Checkpoint guard (same pattern as CUR notebook) ---
def svd_ckpt(name): return f"{SVD_DIR}/{name}"
def out_path(name): return f"{OUTPUT_DIR}/{name}"
def done(path):     return os.path.exists(path)

print("BellKor notebook ready.")


BellKor notebook ready.


---
## Step 1: Load shared CUR artifacts


In [ ]:
# --- Indices ---
user_index = pd.read_csv(f"{CKPT_DIR}/user_index.csv")
movie_index = pd.read_csv(f"{CKPT_DIR}/movie_index.csv")

userId_to_uidx = dict(zip(user_index.userId, user_index.uidx))
uidx_to_userId = dict(zip(user_index.uidx,    user_index.userId))
movieId_to_midx = dict(zip(movie_index.movieId, movie_index.midx))
midx_to_movieId = dict(zip(movie_index.midx,    movie_index.movieId))

n_users  = len(user_index)
n_movies = len(movie_index)

# --- User means + global mean ---
user_means_df = pd.read_csv(f"{CKPT_DIR}/user_means.csv")
uidx_to_mean  = dict(zip(user_means_df.uidx, user_means_df["mean"]))
user_mean_arr = np.array([uidx_to_mean.get(u, 0.0) for u in range(n_users)], dtype=np.float32)

with open(f"{CKPT_DIR}/meta.json") as f:
    meta = json.load(f)
GLOBAL_MEAN = float(meta["global_mean"])
user_mean_arr[user_mean_arr == 0.0] = GLOBAL_MEAN  # cold users -> global mean

# --- Centered training matrix and test set ---
M_centered = load_npz(f"{CKPT_DIR}/M_centered.npz").tocsr()
test_set   = np.load(f"{CKPT_DIR}/test_set.npz")
test_uidx, test_midx, test_rating = test_set["uidx"], test_set["midx"], test_set["rating"]

assert M_centered.shape == (n_users, n_movies), "Index/matrix shape mismatch - CUR checkpoints out of sync."
print(f"  users={n_users:,}  movies={n_movies:,}")
print(f"  M_centered nnz={M_centered.nnz:,}  test={len(test_rating):,}")
print(f"  GLOBAL_MEAN={GLOBAL_MEAN:.4f}")

# Movie titles for top-K demo
movies_meta  = pd.read_csv(MOVIES_PATH, usecols=["movieId", "title"])
title_lookup = dict(zip(movies_meta.movieId, movies_meta.title))


  users=162,541  movies=59,047
  M_centered nnz=20,001,207  test=4,998,888
  GLOBAL_MEAN=3.5339


---
## Step 2: CUR predictor


In [ ]:
C_sp   = load_npz(f"{CKPT_DIR}/C.npz")           # (n_users × r)
R_sp   = load_npz(f"{CKPT_DIR}/R.npz")           # (r × n_movies)
U_long = pd.read_csv(f"{CKPT_DIR}/U.csv")        # CUR notebook cell 1.6
r_dim  = int(max(U_long["row"].max(), U_long["col"].max())) + 1
U_dense = np.zeros((r_dim, r_dim), dtype=np.float32)
U_dense[U_long["row"].values, U_long["col"].values] = U_long["value"].values.astype(np.float32)

R_dense = R_sp.toarray().astype(np.float32)      # (r × n_movies)
UR = U_dense @ R_dense                           # (r × n_movies)
print(f"  UR shape = {UR.shape}, dtype={UR.dtype}, mem={UR.nbytes/1e6:.1f} MB")

def cur_row(uidx):
    c = np.asarray(C_sp[uidx, :].todense()).ravel().astype(np.float32)
    return np.clip(c @ UR + user_mean_arr[uidx], 0.5, 5.0)

# Sanity check
_ = cur_row(0)
print(f"  cur_row(0)[:5] = {cur_row(0)[:5]}")


  UR shape = (500, 59047), dtype=float32, mem=118.1 MB
  cur_row(0)[:5] = [3.8785233 3.8092282 3.8793335 3.8775864 3.875054 ]


---
## Step 3: Train a new SVD model

Using Pradnya's model with everything here would be a bunch of work because I'd need to save the item/user factors, and rerun cur and svd because the test/train splits are different because she used spark and I didn't. This is important because then the blender would overweight ALS on rows where we have training sets being tested.


In [ ]:
SVD_FILES = ["P.npy", "Q.npy", "bi.npy", "meta.json"]

if all(done(svd_ckpt(f)) for f in SVD_FILES):
    print("Step 3: SVD checkpoints found")
    P  = np.load(svd_ckpt("P.npy"))
    Q  = np.load(svd_ckpt("Q.npy"))
    bi = np.load(svd_ckpt("bi.npy"))
else:
    print(f"Step 3: training truncated SVD with k={SVD_RANK} on M_centered")
    # svds returns singular values in ascending order
    U_svd, s, Vt = svds(M_centered.astype(np.float32), k=SVD_RANK)
    order = np.argsort(-s)
    s, U_svd, Vt = s[order], U_svd[:, order], Vt[order, :]
    sqrt_s = np.sqrt(s).astype(np.float32)
    P = (U_svd * sqrt_s).astype(np.float32)        # (n_users, k)
    Q = (Vt.T   * sqrt_s).astype(np.float32)       # (n_movies, k)
    print(f"  P={P.shape}  Q={Q.shape}  top-singular={s[:5].round(3)}")

    # --- Bias polish: per-item bias b_i = mean residual on train, regularized ---
    print("  computing residuals on train and fitting per-item bias...")
    coo = M_centered.tocoo()
    u_arr, m_arr = coo.row.astype(np.int64), coo.col.astype(np.int64)
    centered_vals = coo.data.astype(np.float32)              # = rating - user_mean
    # residual_ui = (rating - user_mean) - P_u · Q_i = centered_val - dot
    dots = np.einsum("ij,ij->i", P[u_arr], Q[m_arr]).astype(np.float32)
    residuals = centered_vals - dots
    sums   = np.bincount(m_arr, weights=residuals, minlength=n_movies)
    counts = np.bincount(m_arr, minlength=n_movies)
    bi = (sums / (counts + SVD_BIAS_LAMBDA)).astype(np.float32)
    del coo, u_arr, m_arr, centered_vals, dots, residuals, sums, counts
    gc.collect()

    np.save(svd_ckpt("P.npy"),  P)
    np.save(svd_ckpt("Q.npy"),  Q)
    np.save(svd_ckpt("bi.npy"), bi)
    with open(svd_ckpt("meta.json"), "w") as f:
        json.dump({"rank": SVD_RANK, "bias_lambda": SVD_BIAS_LAMBDA, "seed": SEED}, f)
    print(f"  saved P, Q, bi to {SVD_DIR}/")

def svd_row(uidx):
    # Predicted rating row for one user (length n_movies), clipped to [0.5, 5.0].
    return np.clip(user_mean_arr[uidx] + bi + P[uidx] @ Q.T, 0.5, 5.0).astype(np.float32)

print(f"  svd_row(0)[:5] = {svd_row(0)[:5]}")


Step 3: SVD checkpoints found
  svd_row(0)[:5] = [3.872333  3.6461203 3.5833905 3.3307903 3.510546 ]


---
## Step 4: Build a (user, item) CBF scorer

Lift content-based filtering from item–item cosine to a user-aware scorer. The user profile is the rating-weighted average of BGE embeddings of items the user rated, computed from sparse `M_centered` matrix. Similar to SVD, redoing this is needed for the blender to work I believe (since it's doing user->item).


In [ ]:
# --- Load BGE embeddings + build midx -> tmdb embedding-row bridge ---
bge = np.load(EMB_PATH).astype(np.float32)
print(f"  BGE embeddings: {bge.shape}")

tmdb_meta = pd.read_csv(TMDB_PATH, usecols=["id", "title"]).rename(columns={"id": "tmdb_id"})
tmdb_meta = tmdb_meta.reset_index().rename(columns={"index": "tmdb_idx"})  # row-index in bge

links = pd.read_csv(LINKS_PATH, usecols=["movieId", "tmdbId"]).dropna()
links["tmdbId"] = links["tmdbId"].astype(np.int64)
bridge = links.merge(tmdb_meta, left_on="tmdbId", right_on="tmdb_id", how="inner")
bridge = bridge.merge(movie_index, on="movieId", how="inner")  # adds midx
midx_to_tmdb_idx = dict(zip(bridge.midx.astype(int), bridge.tmdb_idx.astype(int)))
print(f"  CBF coverage: {len(midx_to_tmdb_idx):,}/{n_movies:,} movies "
      f"({100*len(midx_to_tmdb_idx)/n_movies:.1f}%)")

# --- Build profiles via one sparse-dense matmul ---
covered_midx     = np.array(sorted(midx_to_tmdb_idx.keys()), dtype=np.int64)
embed_covered    = np.array([bge[midx_to_tmdb_idx[m]] for m in covered_midx], dtype=np.float32)
# L2-normalize item embeddings (BGE is already)
embed_covered   /= np.linalg.norm(embed_covered, axis=1, keepdims=True).clip(min=1e-9)

PROFILE_PATH = f"{OUTPUT_DIR}/cbf_user_profiles.npz"
if done(PROFILE_PATH):
    print("Step 4: profiles found")
    z = np.load(PROFILE_PATH)
    profiles = z["profiles"]
    covered_midx = z["covered_midx"]
else:
    print("Step 4: building user profiles...")
    M_covered = M_centered[:, covered_midx]                 # (n_users, n_covered) sparse
    profiles  = (M_covered @ embed_covered).astype(np.float32)   # (n_users, 768)
    norms = np.linalg.norm(profiles, axis=1, keepdims=True)
    profiles = profiles / norms.clip(min=1e-9)
    np.savez_compressed(PROFILE_PATH, profiles=profiles, covered_midx=covered_midx)
    print(f"  saved profiles ({profiles.shape}) to {PROFILE_PATH}")

# Mask: True if movie has an embedding
cbf_mask = np.zeros(n_movies, dtype=bool)
cbf_mask[covered_midx] = True
covered_set = set(covered_midx.tolist())

def cbf_row(uidx):
    # Cosine similarity between user profile and each covered movie; NaN elsewhere.
    out = np.full(n_movies, np.nan, dtype=np.float32)
    out[covered_midx] = profiles[uidx] @ embed_covered.T
    return out

def cbf_pairs(uidxs, midxs):
    # Vectorized CBF score for a batch of (uidx, midx) pairs. NaN if midx uncovered.
    out = np.full(len(uidxs), np.nan, dtype=np.float32)
    in_cov = np.array([m in covered_set for m in midxs], dtype=bool)
    if not in_cov.any():
        return out
    # remap covered midx -> row in embed_covered
    midx_to_row = {int(m): i for i, m in enumerate(covered_midx)}
    rows = np.array([midx_to_row[int(m)] for m in midxs[in_cov]], dtype=np.int64)
    out[in_cov] = np.einsum("ij,ij->i", profiles[uidxs[in_cov]], embed_covered[rows])
    return out

print(f"  cbf_row(0) covered range = "
      f"[{np.nanmin(cbf_row(0)):.3f}, {np.nanmax(cbf_row(0)):.3f}]")


  BGE embeddings: (4809, 768)
  CBF coverage: 4,595/59,047 movies (7.8%)
Step 4: profiles found
  cbf_row(0) covered range = [-0.002, 0.521]


---
## Step 4b: Item-item neighborhood CF (kNN)


We re-use the CUR-derived item embeddings already saved at `cur-output/cur_checkpoints/item_embeddings.npz` those are 500-d item vectors built from rating co-occurrence, so cosine between them is a collaborative (not content) signal. Without this, the blender wasn't very good... Definitely needed this.


In [ ]:
# --- Build top-K item-item neighbor graph from CUR's saved item embeddings ---
import scipy.sparse as sp

ITEM_EMB_PATH  = f"{CKPT_DIR}/item_embeddings.npz"
KNN_GRAPH_PATH = f"{OUTPUT_DIR}/knn_item_graph.npz"
K_NEIGHBORS    = 50
SIM_FLOOR      = 0.0   # negative similarities hurt prediction

if done(KNN_GRAPH_PATH):
    print("Step 4b: kNN graph found - loading.")
    S_knn = sp.load_npz(KNN_GRAPH_PATH).tocsr()
else:
    print(f"Step 4b: building top-{K_NEIGHBORS} item-item neighbor graph from CUR item_embeddings...")
    z = np.load(ITEM_EMB_PATH)
    key = "E" if "E" in z.files else z.files[0]
    E = z[key].astype(np.float32)
    # CUR saved item_embeddings as (r, n_movies)
    if E.shape[1] != n_movies and E.shape[0] == n_movies:
        E = E.T
    print(f"  item_embeddings shape: {E.shape}  (expected (r, {n_movies}))")

    norms = np.linalg.norm(E, axis=0)
    norms = np.maximum(norms, 1e-9)
    E_normed = (E / norms[None, :]).astype(np.float32)

    rows_, cols_, vals_ = [], [], []
    CHUNK = 1000
    for start in range(0, n_movies, CHUNK):
        end = min(start + CHUNK, n_movies)
        # (chunk, r) @ (r, n_movies) -> (chunk, n_movies)
        sim = E_normed[:, start:end].T @ E_normed
        # zero out self-similarity
        sim[np.arange(end - start), np.arange(start, end)] = -np.inf
        # top-K per row
        top = np.argpartition(-sim, K_NEIGHBORS, axis=1)[:, :K_NEIGHBORS]
        for k in range(end - start):
            i = start + k
            neighbors = top[k]
            sims = sim[k, neighbors]
            keep = sims > SIM_FLOOR
            if not keep.any():
                continue
            rows_.extend([i] * int(keep.sum()))
            cols_.extend(neighbors[keep].tolist())
            vals_.extend(sims[keep].tolist())
        if (end // CHUNK) % 10 == 0:
            print(f"  ... {end:,}/{n_movies:,} items")
    S_knn = sp.coo_matrix(
        (np.array(vals_, dtype=np.float32),
         (np.array(rows_, dtype=np.int64), np.array(cols_, dtype=np.int64))),
        shape=(n_movies, n_movies),
    ).tocsr()
    sp.save_npz(KNN_GRAPH_PATH, S_knn)
    print(f"  saved top-{K_NEIGHBORS} graph (nnz={S_knn.nnz:,}) to {KNN_GRAPH_PATH}")
    del E, E_normed, sim
    gc.collect()

# Pre-absolute-value matrix for normalization denominator
S_knn_abs = S_knn.copy()
S_knn_abs.data = np.abs(S_knn_abs.data)

def knn_row(uidx):
    # Item-item CF prediction
    rated = M_centered[uidx, :].T          # (n_movies, 1) sparse, centered ratings
    if rated.nnz == 0:
        return np.full(n_movies, user_mean_arr[uidx], dtype=np.float32)
    numerator   = (S_knn     @ rated).toarray().ravel()         # (n_movies,)
    rated_mask  = rated.copy()
    rated_mask.data = np.ones_like(rated_mask.data, dtype=np.float32)
    denominator = (S_knn_abs @ rated_mask).toarray().ravel()
    safe        = np.where(denominator > 1e-9, denominator, 1.0)
    score       = numerator / safe
    score[denominator < 1e-9] = 0.0
    return np.clip(user_mean_arr[uidx] + score, 0.5, 5.0).astype(np.float32)

print(f"  knn_row(0)[:5] = {knn_row(0)[:5]}")


Step 4b: building top-50 item-item neighbor graph from CUR item_embeddings...
  item_embeddings shape: (500, 59047)  (expected (r, 59047))
  ... 10,000/59,047 items
  ... 20,000/59,047 items
  ... 30,000/59,047 items
  ... 40,000/59,047 items
  ... 50,000/59,047 items
  saved top-50 graph (nnz=1,011,950) to ./cur-output/knn_item_graph.npz
  knn_row(0)[:5] = [3.875 3.875 3.875 3.875 3.875]


---
## Step 4c: Popularity prior



A simple per-item `log(1 + train_count)` feature, min-max scaled. In the blender it lets the model learn an item-level popularity offset that latent-factor models smooth over.


In [ ]:
movie_count = np.diff(M_centered.tocsc().indptr).astype(np.float32)   # train ratings per item
pop_log     = np.log1p(movie_count).astype(np.float32)
# Min-max scale to [0, 1] so the blender weight is on a comparable magnitude to the others
pop_vec_norm = ((pop_log - pop_log.min()) / max(pop_log.max() - pop_log.min(), 1e-9)).astype(np.float32)

def pop_row(uidx):
    # Popularity prior - same vector for every user (user-independent).
    return pop_vec_norm

print(f"  pop_log range: [{pop_log.min():.2f}, {pop_log.max():.2f}]  (log1p of train counts)")
print(f"  median train support per item: {np.median(movie_count):.0f}")


  pop_log range: [0.00, 11.09]  (log1p of train counts)
  median train support per item: 5


---
## Step 5: Carve blend-fit / blend-eval splits



The held-out test set (about 5 M ratings) is split 50/50. Base models (CUR, SVD, CBF) saw none of it during training; the blender weights are learned on `blend_fit` and reported metrics come from `blend_eval`.


In [ ]:
rng_blend = np.random.default_rng(SEED + 1)
mask_fit  = rng_blend.random(len(test_rating)) < 0.5
fit_uidx,  fit_midx,  fit_rating  = test_uidx[mask_fit],  test_midx[mask_fit],  test_rating[mask_fit]
ev_uidx,   ev_midx,   ev_rating   = test_uidx[~mask_fit], test_midx[~mask_fit], test_rating[~mask_fit]
print(f"  blend_fit  = {len(fit_rating):,} ratings")
print(f"  blend_eval = {len(ev_rating):,} ratings")


  blend_fit  = 2,498,851 ratings
  blend_eval = 2,500,037 ratings


---
## Step 6: Score blend-fit triples + fit BellKor blender

For each (u, m) in `blend_fit`, gather predictions from all base models, then
fit a ridge regression with seven features:

`r̂ = w₀ + w_cur·cur + w_svd·svd + w_cbf·cbf_filled + w_flag·cbf_present + w_knn·knn + w_pop·pop`


In [ ]:
def predict_cur_svd_for_pairs(uidxs, midxs):
    # Per-pair CUR + SVD predictions
    cur_preds = np.empty(len(uidxs), dtype=np.float32)
    svd_preds = np.empty(len(uidxs), dtype=np.float32)
    order = np.argsort(uidxs, kind="stable")
    u_s, m_s = uidxs[order], midxs[order]
    unique_users, starts = np.unique(u_s, return_index=True)
    starts = np.append(starts, len(u_s))
    for k, u in enumerate(unique_users):
        s, e = starts[k], starts[k+1]
        cr = cur_row(int(u))
        sr = svd_row(int(u))
        cur_preds[order[s:e]] = cr[m_s[s:e]]
        svd_preds[order[s:e]] = sr[m_s[s:e]]
    return cur_preds, svd_preds

def predict_knn_pop_for_pairs(uidxs, midxs):
    # Per-pair kNN + popularity predictions
    knn_preds = np.empty(len(uidxs), dtype=np.float32)
    pop_preds = pop_vec_norm[midxs].copy()
    order = np.argsort(uidxs, kind="stable")
    u_s, m_s = uidxs[order], midxs[order]
    unique_users, starts = np.unique(u_s, return_index=True)
    starts = np.append(starts, len(u_s))
    for k, u in enumerate(unique_users):
        s, e = starts[k], starts[k+1]
        kr = knn_row(int(u))
        knn_preds[order[s:e]] = kr[m_s[s:e]]
    return knn_preds, pop_preds

print("Scoring blend_fit with CUR + SVD (one row per user)...")
cur_fit, svd_fit = predict_cur_svd_for_pairs(fit_uidx, fit_midx)
print("Scoring blend_fit with kNN + popularity...")
knn_fit, pop_fit = predict_knn_pop_for_pairs(fit_uidx, fit_midx)
print("Scoring blend_fit with CBF (vectorized)...")
cbf_fit_raw = cbf_pairs(fit_uidx, fit_midx)
cbf_present = (~np.isnan(cbf_fit_raw)).astype(np.float32)
cbf_fit     = np.nan_to_num(cbf_fit_raw, nan=0.0).astype(np.float32)

# Feature matrix
X_fit = np.column_stack([
    np.ones(len(fit_rating), dtype=np.float32),
    cur_fit, svd_fit, cbf_fit, cbf_present, knn_fit, pop_fit,
])
y_fit = fit_rating.astype(np.float32)

ridge = Ridge(alpha=RIDGE_ALPHA, fit_intercept=False)
ridge.fit(X_fit, y_fit)
w0, w_cur, w_svd, w_cbf, w_flag, w_knn, w_pop = ridge.coef_.tolist()
r2 = ridge.score(X_fit, y_fit)
cov = float(cbf_present.mean())

blender = {
    "w0": w0, "w_cur": w_cur, "w_svd": w_svd, "w_cbf": w_cbf, "w_flag": w_flag,
    "w_knn": w_knn, "w_pop": w_pop,
    "ridge_alpha": RIDGE_ALPHA, "blend_fit_size": int(len(y_fit)),
    "cbf_coverage_in_blend_fit": cov, "train_R2": float(r2),
}
with open(out_path("bellkor_blender.json"), "w") as f:
    json.dump(blender, f, indent=2)

print("\nBellKor blender weights:")
for k, v in blender.items():
    print(f"  {k:30s} = {v:.4f}" if isinstance(v, float) else f"  {k:30s} = {v}")


Scoring blend_fit with CUR + SVD (one row per user)...
Scoring blend_fit with kNN + popularity...
Scoring blend_fit with CBF (vectorized)...

BellKor blender weights:
  w0                             = 0.2159
  w_cur                          = -0.2768
  w_svd                          = 1.1511
  w_cbf                          = 0.1542
  w_flag                         = -0.0017
  w_knn                          = 0.0192
  w_pop                          = 0.2281
  ridge_alpha                    = 1.0000
  blend_fit_size                 = 2498851
  cbf_coverage_in_blend_fit      = 0.6907
  train_R2                       = 0.3748


---
## Step 7: Side-by-side evaluation

Same protocol as `CUR.ipynb` cells 20 & 22, run four times: RMSE / MAE on full `blend_eval`, then Precision@K, Recall@K, HitRate@K, NDCG@K on a sampled set of users.


In [ ]:
# --- 7a: RMSE / MAE on blend_eval for each model ---
print("Scoring blend_eval with CUR + SVD...")
cur_ev, svd_ev = predict_cur_svd_for_pairs(ev_uidx, ev_midx)
print("Scoring blend_eval with kNN + popularity...")
knn_ev, pop_ev = predict_knn_pop_for_pairs(ev_uidx, ev_midx)
print("Scoring blend_eval with CBF...")
cbf_ev_raw = cbf_pairs(ev_uidx, ev_midx)
cbf_present_ev = (~np.isnan(cbf_ev_raw)).astype(np.float32)
cbf_ev = np.nan_to_num(cbf_ev_raw, nan=0.0).astype(np.float32)

# For CBF-only RMSE we map cosine->rating via a 1-feature ridge fit on blend_fit.
ridge_cbf = Ridge(alpha=1.0, fit_intercept=True)
mask_cov_fit = cbf_present.astype(bool)
ridge_cbf.fit(cbf_fit[mask_cov_fit].reshape(-1, 1), y_fit[mask_cov_fit])
cbf_only_ev = np.where(cbf_present_ev.astype(bool),
                       ridge_cbf.predict(cbf_ev.reshape(-1, 1)),
                       GLOBAL_MEAN).astype(np.float32)

bell_ev = (w0 + w_cur*cur_ev + w_svd*svd_ev + w_cbf*cbf_ev + w_flag*cbf_present_ev
           + w_knn*knn_ev + w_pop*pop_ev).astype(np.float32)
bell_ev = np.clip(bell_ev, 0.5, 5.0)

def rmse_mae(pred, target):
    err = pred - target
    return float(np.sqrt(np.mean(err**2))), float(np.mean(np.abs(err)))

rows = []
for name, pred in [("CUR",                 cur_ev),
                   ("SVD",                 svd_ev),
                   ("CBF (rating-scaled)", cbf_only_ev),
                   ("kNN item-item CF",    knn_ev),
                   ("BellKor blend",       bell_ev)]:
    r, m = rmse_mae(pred, ev_rating)
    rows.append({"model": name, "RMSE": r, "MAE": m})
rmse_df = pd.DataFrame(rows)
print("\n--- RMSE / MAE on blend_eval ({:,} ratings) ---".format(len(ev_rating)))
print(rmse_df.to_string(index=False))


Scoring blend_eval with CUR + SVD...
Scoring blend_eval with kNN + popularity...
Scoring blend_eval with CBF...

--- RMSE / MAE on blend_eval (2,500,037 ratings) ---
              model     RMSE      MAE
                CUR 0.957181 0.740140
                SVD 0.844945 0.643423
CBF (rating-scaled) 1.060192 0.840367
   kNN item-item CF 1.084373 0.816010
      BellKor blend 0.838466 0.637861


In [ ]:
# --- 7b: Ranking metrics on a user sample ---
from collections import defaultdict

# Eligibility mask: items with enough train support, in CUR-style
movie_support = np.diff(M_centered.tocsc().indptr)        # ratings per movie
eligible_mask_score = np.where(movie_support >= MIN_SUPPORT, 0.0, -np.inf).astype(np.float32)
eligible_pool = np.where(eligible_mask_score == 0.0)[0]

M_train_csr = M_centered  # already CSR
def seen_midx_for(uidx):
    return M_train_csr[uidx, :].indices

# Build (user -> relevant items) from blend_eval positives only
relevant = defaultdict(set)
for u, m, r in zip(ev_uidx, ev_midx, ev_rating):
    if r >= RELEVANCE_THRESHOLD:
        relevant[int(u)].add(int(m))
eligible_users = [u for u, rel in relevant.items() if len(rel) >= MIN_RELEVANT]
print(f"  eligible eval users: {len(eligible_users):,}")

rng_eval = np.random.default_rng(SEED + 2)
sample_users = rng_eval.choice(eligible_users,
                               size=min(EVAL_USERS, len(eligible_users)),
                               replace=False)
sample_users = sorted(int(u) for u in sample_users)

def eval_scorer(name, score_row_fn):
    # Compute Precision@K, Recall@K, HitRate@K, NDCG@K for a per-user score function.
    log2_pos = 1.0 / np.log2(np.arange(2, K + 2))
    precisions, recalls, hits, ndcgs = [], [], [], []
    n_full, n_samp = 0, 0
    user_to_pos = defaultdict(list)
    for u, m, r in zip(ev_uidx, ev_midx, ev_rating):
        if r >= RELEVANCE_THRESHOLD:
            user_to_pos[int(u)].append(int(m))

    for u in sample_users:
        rel = relevant[u]
        scores = score_row_fn(u) + eligible_mask_score
        scores[seen_midx_for(u)] = -np.inf

        # Full Precision@K / Recall@K
        topk = np.argpartition(-scores, K)[:K]
        hit_full = len(set(topk.tolist()) & rel)
        precisions.append(hit_full / K)
        recalls.append(hit_full / len(rel))
        n_full += 1

        # Sampled HitRate@K / NDCG@K
        seen = set(seen_midx_for(u).tolist()) | rel
        for pos in user_to_pos[u]:
            negs = []
            attempts = 0
            while len(negs) < NUM_NEGATIVES and attempts < 5:
                cand = rng_eval.choice(eligible_pool, size=NUM_NEGATIVES * 2)
                negs.extend(int(x) for x in cand if int(x) not in seen)
                attempts += 1
            negs = negs[:NUM_NEGATIVES]
            if len(negs) < NUM_NEGATIVES:
                continue
            cand = np.array([pos] + negs, dtype=np.int64)
            cs = scores[cand] if not np.any(np.isneginf(scores[cand])) else None
            if cs is None or not np.isfinite(scores[pos]):
                bare = score_row_fn(u)
                cs = bare[cand]
            rank = int((cs > cs[0]).sum())
            if rank < K:
                hits.append(1.0); ndcgs.append(log2_pos[rank])
            else:
                hits.append(0.0); ndcgs.append(0.0)
            n_samp += 1
    return {
        "model": name,
        "Precision@10": float(np.mean(precisions)),
        "Recall@10":    float(np.mean(recalls)),
        "HitRate@10":   float(np.mean(hits)),
        "NDCG@10":      float(np.mean(ndcgs)),
        "n_users":      n_full,
        "n_pairs":      n_samp,
    }

def bell_row(uidx):
    cr = cur_row(uidx); sr = svd_row(uidx); cb = cbf_row(uidx)
    kr = knn_row(uidx); pr = pop_row(uidx)
    cb_pres = (~np.isnan(cb)).astype(np.float32)
    cb_filled = np.nan_to_num(cb, nan=0.0)
    return np.clip(w0 + w_cur*cr + w_svd*sr + w_cbf*cb_filled + w_flag*cb_pres
                   + w_knn*kr + w_pop*pr,
                   0.5, 5.0).astype(np.float32)

def cbf_only_row(uidx):
    cb = cbf_row(uidx)
    return np.where(np.isnan(cb), GLOBAL_MEAN, cb).astype(np.float32)

print("\nRanking eval - this takes a few minutes per model...")
rank_rows = []
for name, fn in [("CUR",              cur_row),
                 ("SVD",              svd_row),
                 ("CBF",              cbf_only_row),
                 ("kNN item-item CF", knn_row),
                 ("Popularity-only",  pop_row),
                 ("BellKor blend",    bell_row)]:
    print(f"  {name}...")
    rank_rows.append(eval_scorer(name, fn))
rank_df = pd.DataFrame(rank_rows)
print("\n--- Ranking metrics on {:,} sampled users ---".format(len(sample_users)))
print(rank_df.to_string(index=False))

# Persist combined eval
combined = {
    "rmse_mae":        rmse_df.to_dict(orient="records"),
    "ranking":         rank_df.to_dict(orient="records"),
    "blend_fit_size":  int(len(y_fit)),
    "blend_eval_size": int(len(ev_rating)),
    "ranking_users":   len(sample_users),
}
with open(out_path("bellkor_eval.json"), "w") as f:
    json.dump(combined, f, indent=2)
print(f"\nSaved combined eval to {out_path('bellkor_eval.json')}")


  eligible eval users: 127,870

Ranking eval — this takes a few minutes per model...
  CUR...
  SVD...
  CBF...
  kNN item-item CF...
  Popularity-only...
  BellKor blend...

--- Ranking metrics on 5,000 sampled users ---
           model  Precision@10  Recall@10  HitRate@10  NDCG@10  n_users  n_pairs
             CUR       0.01282   0.015499    0.281658 0.171010     5000    47650
             SVD       0.02886   0.022099    0.328898 0.202436     5000    47650
             CBF       0.00126   0.001928    0.293054 0.293054     5000    47650
kNN item-item CF       0.00186   0.002762    0.505897 0.258691     5000    47650
 Popularity-only       0.07266   0.097608    0.805184 0.543076     5000    47650
   BellKor blend       0.03798   0.032558    0.419559 0.259158     5000    47650

Saved combined eval to ./cur-output/bellkor_eval.json


---
## Step 8: Top-n demo with per-model contribution breakdown

For a few sample users, show the BellKor top-10 alongside the score from each component model. (honestly a sanity check)


In [ ]:
SAMPLE_USERS = [1, 42, 100]   # match CineFusion CUR.ipynb spirit

def top10_breakdown(user_id, top_n=10):
    if user_id not in userId_to_uidx:
        return None
    u = userId_to_uidx[user_id]
    cr = cur_row(u); sr = svd_row(u); cb = cbf_row(u)
    kr = knn_row(u); pr = pop_row(u)
    cb_pres = (~np.isnan(cb)).astype(np.float32)
    cb_filled = np.nan_to_num(cb, nan=0.0)
    blend = np.clip(w0 + w_cur*cr + w_svd*sr + w_cbf*cb_filled + w_flag*cb_pres
                    + w_knn*kr + w_pop*pr,
                    0.5, 5.0)
    blend = blend + eligible_mask_score
    blend[seen_midx_for(u)] = -np.inf
    top = np.argpartition(-blend, top_n)[:top_n]
    top = top[np.argsort(-blend[top])]
    return pd.DataFrame({
        "title":   [title_lookup.get(midx_to_movieId[int(m)], "?") for m in top],
        "movieId": [int(midx_to_movieId[int(m)]) for m in top],
        "cur":     [round(float(cr[m]), 3) for m in top],
        "svd":     [round(float(sr[m]), 3) for m in top],
        "cbf":     [round(float(cb[m]) if not np.isnan(cb[m]) else float("nan"), 3) for m in top],
        "knn":     [round(float(kr[m]), 3) for m in top],
        "pop":     [round(float(pr[m]), 3) for m in top],
        "blended": [round(float(blend[m]), 3) for m in top],
    })

for uid in SAMPLE_USERS:
    print(f"\n=== Top-10 BellKor recommendations for userId={uid} ===")
    df = top10_breakdown(uid, 5)
    if df is None:
        print("  (user not in training set)")
    else:
        print(df.to_string(index=False))



=== Top-10 BellKor recommendations for userId=1 ===
                  title  movieId   cur   svd  cbf   knn   pop  blended
    Planet Earth (2006)   159817 3.845 4.652  NaN 3.875 0.655    4.730
 Planet Earth II (2016)   171011 3.858 4.660  NaN 3.875 0.613    4.726
Band of Brothers (2001)   170705 3.880 4.618  NaN 3.875 0.632    4.675
                 Cosmos   171495 3.876 4.569  NaN 3.875 0.488    4.587
      Twin Peaks (1989)   198185 3.875 4.529  NaN 3.875 0.492    4.543

=== Top-10 BellKor recommendations for userId=42 ===
                  title  movieId   cur   svd  cbf   knn   pop  blended
    Planet Earth (2006)   159817 3.779 4.554  NaN 3.778 0.655    4.633
 Planet Earth II (2016)   171011 3.777 4.560  NaN 3.778 0.613    4.632
Band of Brothers (2001)   170705 3.780 4.516  NaN 3.778 0.632    4.584
                 Cosmos   171495 3.779 4.472  NaN 3.778 0.488    4.501
      Twin Peaks (1989)   198185 3.778 4.432  NaN 3.778 0.492    4.457

=== Top-10 BellKor recommendations for u

### I'm working on this right now.

Here, kNN is defaulting because the user hasn't rated any of these movies, cbf can't contribute, and svd is dominating the recommendation. It has to do with the per-item bias (movies like Planet Earth are rated very very high on average).

---
## Step 8b: "Movies similar to X" item-item lookup

Same blending philosophy as the rest of the notebook, but at the *item-item*
similarity level. Three signals:

- **CF co-rating** - true item-item cosine on the centered rating matrix `M`
- **SVD item-item** - cosine on the `Q` factor matrix. Captures broad taste
  clusters but mixes in unrelated co-watched titles together.
- **CBF** - cosine on BAAI/BGE content embeddings (only if seed has a TMDB embedding).


In [ ]:
def _resolve_movie(query):
    # Exact title match first, then case-insensitive substring. Returns matching rows in movies_meta.
    q = str(query).strip()
    exact = movies_meta[movies_meta.title.str.lower() == q.lower()]
    cand = exact if len(exact) > 0 else movies_meta[
        movies_meta.title.str.lower().str.contains(q.lower(), regex=False, na=False)
    ]
    return cand[cand.movieId.isin(movieId_to_midx)]

# One-time precompute: per-item L2 norm on the centered rating matrix ---
M_csc = M_centered.tocsc()
_item_l2 = np.sqrt(np.array(M_csc.multiply(M_csc).sum(axis=0)).ravel()).astype(np.float32)
_item_l2 = np.maximum(_item_l2, 1e-9)
_q_norms = np.maximum(np.linalg.norm(Q, axis=1).astype(np.float32), 1e-9)
print(f"  precomputed item L2 norms (median support-weighted norm = {np.median(_item_l2):.2f})")

def similar_movies(query, top_n=10, weights=(0.6, 0.1, 0.3), min_support=20):
    matches = _resolve_movie(query)
    if len(matches) == 0:
        print(f"No movie matching {query!r}."); return None
    seed = matches.iloc[0]
    seed_midx = movieId_to_midx[seed.movieId]
    if len(matches) > 1:
        others = matches.iloc[1:6].title.tolist()
        print(f"Seed: {seed.title}  (matched {len(matches)} titles; also: {others})")
    else:
        print(f"Seed: {seed.title}")

    # 1. CF co-rating cosine (the important one for "more like this")
    #    sim(seed, j) = <M[:, seed], M[:, j]> / (||M[:, seed]|| * ||M[:, j]||)
    seed_col = M_csc[:, seed_midx]                       # (n_users, 1) sparse
    dots     = (M_csc.T @ seed_col).toarray().ravel()    # (n_movies,)
    cf_sims  = (dots / (_item_l2 * _item_l2[seed_midx])).astype(np.float32)

    # 2. SVD item-item cosine on Q
    q_seed   = Q[seed_midx]
    svd_sims = ((Q @ q_seed) / (_q_norms * max(float(np.linalg.norm(q_seed)), 1e-9))).astype(np.float32)

    # 3. CBF cosine on BGE - only if seed has a TMDB embedding
    cbf_sims = np.full(n_movies, np.nan, dtype=np.float32)
    seed_has_cbf = seed_midx in midx_to_tmdb_idx
    if seed_has_cbf:
        seed_emb = bge[midx_to_tmdb_idx[seed_midx]]
        seed_emb = seed_emb / max(float(np.linalg.norm(seed_emb)), 1e-9)
        cbf_sims[covered_midx] = embed_covered @ seed_emb
    cbf_present_mask = ~np.isnan(cbf_sims)
    cbf_filled = np.nan_to_num(cbf_sims, nan=0.0)

    # Blend (raw cosines - no min-max normalization that would amplify noise)
    w_cf, w_svd_, w_cbf_ = weights
    blended = w_cf * cf_sims + w_svd_ * svd_sims + w_cbf_ * cbf_filled

    # Filter: require minimum train support (suppress noisy obscure items)
    support_mask = movie_support >= min_support
    blended[~support_mask] = -np.inf
    blended[seed_midx]     = -np.inf

    top = np.argpartition(-blended, top_n)[:top_n]
    top = top[np.argsort(-blended[top])]
    return pd.DataFrame({
        "title":     [title_lookup.get(midx_to_movieId[int(m)], "?") for m in top],
        "movieId":   [int(midx_to_movieId[int(m)]) for m in top],
        "n_ratings": [int(movie_support[m]) for m in top],
        "cf_corate": [round(float(cf_sims[m]), 3)  for m in top],
        "svd_cos":   [round(float(svd_sims[m]), 3) for m in top],
        "cbf_cos":   [round(float(cbf_sims[m]) if cbf_present_mask[m] else float("nan"), 3) for m in top],
        "blended":   [round(float(blended[m]), 3) for m in top],
    })

# --- Examples ---
print("\n=== Similar to Toy Story (1995) ===")
print(similar_movies("Toy Story (1995)", top_n=10).to_string(index=False))

#print("\n=== Similar to The Matrix ===")
#print(similar_movies("Matrix, The (1999)", top_n=10).to_string(index=False))

#print("\n=== Similar to Inception, content-heavy (cbf=0.6) ===")
#print(similar_movies("Inception", top_n=10, weights=(0.3, 0.1, 0.6)).to_string(index=False))

#print("\n=== Pure co-rating (cf=1.0) - strict 'people who watched X also watched' ===")
#print(similar_movies("Toy Story (1995)", top_n=10, weights=(1.0, 0.0, 0.0)).to_string(index=False))


  precomputed item L2 norms (median support-weighted norm = 2.03)

=== Similar to Toy Story (1995) ===
Seed: Toy Story (1995)
                           title  movieId  n_ratings  cf_corate  svd_cos  cbf_cos  blended
              Toy Story 2 (1999)     3114      21259      0.298    0.850    0.593    0.442
              Toy Story 3 (2010)    78499      11484      0.176    0.599    0.626    0.354
           Monsters, Inc. (2001)     4886      27544      0.196    0.394    0.544    0.320
             Finding Nemo (2003)     6377      27844      0.174    0.334    0.582    0.312
            Bug's Life, A (1998)     2355      18004      0.134    0.572    0.538    0.299
         Incredibles, The (2004)     8961      24452      0.153    0.291    0.567    0.291
Shawshank Redemption, The (1994)      318      65088      0.128    0.032    0.668    0.281
           Lion King, The (1994)      364      34153      0.161    0.110    0.569    0.278
                      Big (1988)     2797      16415   

---
## Step 8c: Per-model similar-movies (no blending)

Same seed, four separate top-10 lists. One per base model to see what each signal contributes


In [ ]:
# --- Load CUR item embeddings (full, not the top-50 sparse graph) ---
_cur_z   = np.load(f"{CKPT_DIR}/item_embeddings.npz")
_cur_key = "E" if "E" in _cur_z.files else _cur_z.files[0]
E_cur    = _cur_z[_cur_key].astype(np.float32)
if E_cur.shape[1] != n_movies and E_cur.shape[0] == n_movies:
    E_cur = E_cur.T                                          # ensure (r, n_movies)
_cur_norms   = np.maximum(np.linalg.norm(E_cur, axis=0), 1e-9)
E_cur_normed = (E_cur / _cur_norms[None, :]).astype(np.float32)
print(f"  CUR item embeddings: {E_cur.shape}, mem={E_cur.nbytes/1e6:.0f} MB")

def _topk_table(sim_col_name, sims, seed_midx, top_n, min_support):
    s = sims.copy()
    s[movie_support < min_support] = -np.inf
    s[seed_midx] = -np.inf
    s[~np.isfinite(s)] = -np.inf
    top = np.argpartition(-s, top_n)[:top_n]
    top = top[np.argsort(-s[top])]
    return pd.DataFrame({
        "rank":      list(range(1, top_n + 1)),
        "title":     [title_lookup.get(midx_to_movieId[int(m)], "?") for m in top],
        "n_ratings": [int(movie_support[m]) for m in top],
        sim_col_name:[round(float(sims[m]) if np.isfinite(sims[m]) else float("nan"), 3) for m in top],
    })

def similar_per_model(query, top_n=10, min_support=20):
    """Print top-N similar movies under each base model independently."""
    matches = _resolve_movie(query)
    if len(matches) == 0:
        print(f"No movie matching {query!r}."); return None
    seed = matches.iloc[0]
    seed_midx = movieId_to_midx[seed.movieId]
    print("=" * 80)
    print(f"SEED: {seed.title}  (movieId={seed.movieId}, "
          f"train ratings={int(movie_support[seed_midx])})")
    print("=" * 80)

    # 1. CF co-rating - direct cosine on centered rating matrix
    seed_col = M_csc[:, seed_midx]
    dots     = (M_csc.T @ seed_col).toarray().ravel()
    cf_sims  = (dots / (_item_l2 * _item_l2[seed_midx])).astype(np.float32)

    # 2. CUR - cosine on full CUR item embeddings
    cur_seed = E_cur_normed[:, seed_midx]
    cur_sims = (E_cur_normed.T @ cur_seed).astype(np.float32)

    # 3. SVD - cosine on Q
    q_seed   = Q[seed_midx]
    svd_sims = ((Q @ q_seed) / (_q_norms * max(float(np.linalg.norm(q_seed)), 1e-9))).astype(np.float32)

    # 4. CBF - cosine on BGE (only candidates with embeddings; -inf elsewhere)
    cbf_sims = np.full(n_movies, -np.inf, dtype=np.float32)
    seed_has_cbf = seed_midx in midx_to_tmdb_idx
    if seed_has_cbf:
        seed_emb = bge[midx_to_tmdb_idx[seed_midx]]
        seed_emb = seed_emb / max(float(np.linalg.norm(seed_emb)), 1e-9)
        cbf_sims[covered_midx] = embed_covered @ seed_emb

    out = {}
    for label, sims in [("cf_corate_cos", cf_sims),
                        ("cur_cos",       cur_sims),
                        ("svd_cos",       svd_sims),
                        ("cbf_cos",       cbf_sims)]:
        model_name = label.split("_cos")[0]
        if model_name == "cbf" and not seed_has_cbf:
            print(f"\n--- CBF ---  (skipped: seed has no TMDB/BGE embedding)")
            continue
        df = _topk_table(label, sims, seed_midx, top_n, min_support)
        out[model_name] = df
        print(f"\n--- {model_name.upper()} ---")
        print(df.to_string(index=False))
    return out

# --- Examples ---
similar_per_model("Toy Story (1995)",  top_n=10)


  CUR item embeddings: (500, 59047), mem=118 MB
SEED: Toy Story (1995)  (movieId=1, train ratings=45911)

--- CF_CORATE ---
 rank                                                                          title  n_ratings  cf_corate_cos
    1                                                             Toy Story 2 (1999)      21259          0.298
    2                                                          Monsters, Inc. (2001)      27544          0.196
    3                                                             Toy Story 3 (2010)      11484          0.176
    4                                                            Finding Nemo (2003)      27844          0.174
    5                                                          Lion King, The (1994)      34153          0.161
    6                                                                 Aladdin (1992)      34716          0.156
    7                                                        Incredibles, The (2004)      24452    

{'cf_corate':    rank                                              title  n_ratings  \
 0     1                                 Toy Story 2 (1999)      21259   
 1     2                              Monsters, Inc. (2001)      27544   
 2     3                                 Toy Story 3 (2010)      11484   
 3     4                                Finding Nemo (2003)      27844   
 4     5                              Lion King, The (1994)      34153   
 5     6                                     Aladdin (1992)      34716   
 6     7                            Incredibles, The (2004)      24452   
 7     8                          Back to the Future (1985)      39743   
 8     9  Raiders of the Lost Ark (Indiana Jones and the...      43810   
 9    10          Star Wars: Episode IV - A New Hope (1977)      55038   
 
    cf_corate_cos  
 0          0.298  
 1          0.196  
 2          0.176  
 3          0.174  
 4          0.161  
 5          0.156  
 6          0.153  
 7         